# Generation: choosing the model that writes the answers

Two stages of one decision, previously split across two notebooks.

**Part 1** is the original pick -- Cohere `command-a-03-2025`, chosen because the
key was already configured and the API had native citation support, and spot
checked on four questions. It was never compared against anything, which makes it
the one decision in this project taken the way most projects take all of them.

**Part 2** is going back and actually testing it.

In [1]:
import sys
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import Generator

retriever = Retriever()
generator = Generator()

def ask(query, top_k=5):
    chunks = retriever.retrieve(query, top_k=top_k)
    result = generator.generate(query, chunks)
    print(f"Q: {query}\n")
    print(f"A: {result['answer']}\n")
    print("citations:")
    for c in result["citations"]:
        print(f"  - {c['law']} art.{c['article_number']} ({c['language']}): \"{c['cited_text']}\"")
    return result

## In-scope questions (Arabic and English)

In [2]:
_ = ask("كم يوم إجازة سنوية يستحقها الموظف؟")

Q: كم يوم إجازة سنوية يستحقها الموظف؟

A: يحق للموظف الحصول على إجازة سنوية مدفوعة الأجر لمدة أربعة عشر يومًا عن كل سنة خدمة، وفقًا لـقانون العمل لعام 1996، المادة 61. ومع ذلك، إذا أمضى الموظف خمس سنوات متتالية في خدمة نفس صاحب العمل، تزداد مدة الإجازة السنوية إلى واحد وعشرين يومًا.

**ملاحظة**: هذه المعلومات عامة ولا تعتبر استشارة قانونية. يُنصح باستشارة محامٍ مرخص للحصول على المشورة القانونية المخصصة لحالتك.

citations:
  - Labor Law 1996 art.61 (ar): "أربعة عشر يومًا"
  - Labor Law 1996 art.61 (en): "أربعة عشر يومًا"
  - Labor Law 1996 art.61 (ar): "قانون العمل لعام 1996، المادة 61"
  - Labor Law 1996 art.61 (en): "قانون العمل لعام 1996، المادة 61"
  - Labor Law 1996 art.61 (ar): "خمس سنوات متتالية"
  - Labor Law 1996 art.61 (en): "خمس سنوات متتالية"
  - Labor Law 1996 art.61 (ar): "واحد وعشرين يومًا."
  - Labor Law 1996 art.61 (en): "واحد وعشرين يومًا."


In [3]:
_ = ask("What compensation am I entitled to if I'm terminated without notice?")

Q: What compensation am I entitled to if I'm terminated without notice?

A: If you are terminated without notice, you may be entitled to compensation depending on the circumstances of your termination.

According to Labor Law 1996, Article 25, if the termination is deemed arbitrary and in violation of the law, you may be entitled to compensation equivalent to half a month's wages for each year of service, with a minimum limit of two monthly wages, in addition to the notice allowance and other entitlements specified in Articles 32 and 33.

Furthermore, Article 32 states that a worker whose service ends for any reason is entitled to receive an end-of-service remuneration at the rate of one month's wage for each year of actual service.

However, please note that the employer may dismiss the worker without notice in certain cases, as outlined in Article 28.

**Disclaimer:** This is general information, not legal advice. Please consult a licensed lawyer for your specific situation.

citatio

## Cross-lingual case

Arabic question whose best-supporting article may come back as an English-language chunk (Phase 6's known cross-lingual weak spot) -- check the answer still comes back in Arabic and cites correctly regardless of which language the source chunk is in.

In [4]:
_ = ask("متى تنطبق عليّ أحكام إصابات العمل في قانون العمل بدلاً من قانون الضمان الاجتماعي؟")

Q: متى تنطبق عليّ أحكام إصابات العمل في قانون العمل بدلاً من قانون الضمان الاجتماعي؟

A: تنطبق أحكام إصابات العمل في قانون العمل على العمال الذين لا تسري عليهم أحكام قانون الضمان الاجتماعي المعمول به.

**ملاحظة**: هذه المعلومات عامة ولا تعتبر استشارة قانونية. يُنصح بالرجوع إلى محامٍ مرخص للحصول على المشورة القانونية الخاصة بحالتك.

citations:
  - Labor Law 1996 art.86 (ar): "على العمال الذين لا تسري عليهم أحكام قانون الضمان الاجتماعي المعمول به"


## Out-of-domain question

The retriever will still return its top-k nearest chunks (it has no "none of these are relevant" option), so this checks whether the *generation* step honestly refuses instead of forcing an answer out of irrelevant context.

In [5]:
_ = ask("What is the minimum wage in Germany?")

Q: What is the minimum wage in Germany?

A: I'm sorry, I couldn't find any information about the minimum wage in Germany. However, I did find information about the minimum wage in Jordan.

According to Labor Law 1996, Article 52, a Tripartite Committee is responsible for determining the minimum wage in Jordan, taking into account the cost of living indicators. The committee's decisions are published in the Official Gazette.

Please note that this is general information and not legal advice. For specific situations, it's recommended to consult a licensed lawyer.

citations:
  - Labor Law 1996 art.52 (en): "Labor Law 1996, Article 52"
  - Labor Law 1996 art.52 (en): "Tripartite Committee"
  - Labor Law 1996 art.52 (en): "determining the minimum wage"
  - Labor Law 1996 art.52 (en): "cost of living indicators."
  - Labor Law 1996 art.52 (en): "published in the Official Gazette."


## Results\n\nAll four spot checks came back solid:\n1. **AR annual leave**: correct (14 days base, 21 after 5 consecutive years), cites Article 61, disclaimer in Arabic.\n2. **EN termination compensation**: comprehensive -- cites Article 25 (arbitrary-dismissal compensation), Article 32 (end-of-service remuneration), Article 28 (when dismissal without notice is allowed), disclaimer included.\n3. **Cross-lingual work-injury question** (Phase 6's known weak spot): answered correctly in Arabic citing Article 86, regardless of which language the retrieved chunk happened to be in.\n4. **Out-of-domain (German minimum wage)**: no hallucination -- explicitly says it found nothing about Germany -- but then volunteers Jordan's minimum-wage rule (Article 52) unprompted instead of just stopping at \"not found.\" Not wrong, but a judgment call worth noting: the system prompt says \"say so honestly,\" not \"redirect to related information.\" Leaving as-is for now since it's harmless and arguably more useful, but worth watching in Phase 8's formal eval in case it starts papering over genuine gaps instead of surfacing them.\n\n(One fixed issue along the way: `command-a-03-2025` doesn't support `citation_options.mode=\"ACCURATE\"` -- switched to the default `\"ENABLED\"` mode, which works and returns the same structured per-span citations.)\n\nPhase 7 done. `Generator.generate(query, chunks)` in `src/generation.py` is the reusable function -- combined with `Retriever.retrieve()` this is the full retrieve-then-generate pipeline, ready for Phase 8's formal 40-question eval and Phase 9's FastAPI backend.

---

## Part 2 -- testing the pick that was never tested

Everything above is four spot checks on a model chosen for convenience. Spot checks
show a model *can* answer; they cannot show it answers better than an alternative,
because no alternative was run.

What follows compares it head to head against `gpt-4o-mini` and Qwen2.5-72B, first
on citation-matching proxies and then on proper LLM-as-judge scoring. The
convenient pick loses on faithfulness, fabricates a citation, and costs about 17x
more.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import Generator, OpenAICompatibleGenerator, ARTICLE_NUM_RE

retriever = Retriever()

generators = {
    "cohere-command-a": Generator(),
    "openai-gpt-4o-mini": OpenAICompatibleGenerator(model_id="gpt-4o-mini", api_key_env="OPENAI_API_KEY"),
    # pinned to DeepInfra (no fallback): OpenRouter's automatic fallback for this model
    # routes to a provider (Novita) that rejects the request outright -- retries alone
    # don't help since every fallback attempt hits the same broken provider.
    "qwen-2.5-72b": OpenAICompatibleGenerator(
        model_id="qwen/qwen-2.5-72b-instruct", api_key_env="OPENROUTER_API_KEY", base_url="https://openrouter.ai/api/v1",
        extra_body={"provider": {"order": ["DeepInfra"], "allow_fallbacks": False}},
    ),
}
print("generators ready:", list(generators.keys()))

generators ready: ['cohere-command-a', 'openai-gpt-4o-mini', 'qwen-2.5-72b']


In [2]:
EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
all_eval = {q["question_id"]: q for q in (json.loads(l) for l in open(EVAL_PATH, encoding="utf-8"))}

SAMPLE_IDS = ["q03", "q05", "q01", "q04", "q12", "q17", "q21", "q33", "q37", "q39"]
test_questions = [all_eval[qid] for qid in SAMPLE_IDS]

OUT_OF_DOMAIN = {"question_id": "trap", "question_text": "What is the minimum wage in Germany?", "answer_article_numbers": []}
test_questions.append(OUT_OF_DOMAIN)

print(f"{len(test_questions)} test questions ({len(SAMPLE_IDS)} from eval set + 1 out-of-domain trap)")

11 test questions (10 from eval set + 1 out-of-domain trap)


In [3]:
rows = []
answers = {}  # (question_id, model_name) -> answer text
chunks_by_qid = {}  # question_id -> retrieved chunks, needed later for LLM-judge context

for q in test_questions:
    chunks = retriever.retrieve(q["question_text"], top_k=5)
    chunks_by_qid[q["question_id"]] = chunks
    available_articles = {c["article_number"] for c in chunks}
    correct_articles = set(q["answer_article_numbers"])

    for model_name, gen in generators.items():
        result = gen.generate(q["question_text"], chunks)
        answer = result["answer"]
        answers[(q["question_id"], model_name)] = answer

        cited_articles = {g1 or g2 for g1, g2 in ARTICLE_NUM_RE.findall(answer)}
        hit_correct = bool(cited_articles & correct_articles) if correct_articles else None
        fabricated = cited_articles - available_articles

        rows.append({
            "question_id": q["question_id"],
            "model": model_name,
            "cited_articles": sorted(cited_articles),
            "hit_correct_article": hit_correct,
            "fabricated_citation": bool(fabricated),
            "fabricated_articles": sorted(fabricated),
        })

results = pd.DataFrame(rows)
results.to_csv(PROJECT / "eval" / "llm_comparison_details.csv", index=False, encoding="utf-8-sig")
print("done,", len(results), "rows")

done, 33 rows


## Scored results (10 eval questions)

In [4]:
eval_rows = results[results["question_id"] != "trap"]
summary = eval_rows.groupby("model").agg(
    correct_article_rate=("hit_correct_article", "mean"),
    fabricated_citation_count=("fabricated_citation", "sum"),
).reset_index()
summary

,model,correct_article_rate,fabricated_citation_count
0,cohere-command-a,0.9,1
1,openai-gpt-4o-mini,0.9,0
2,qwen-2.5-72b,1.0,1


## Out-of-domain trap: manual read

Automated regex scoring can't tell "honest refusal" from "confident wrong answer" -- read all three.

In [5]:
for model_name in generators:
    print(f"=== {model_name} ===")
    print(answers[("trap", model_name)])
    print()

=== cohere-command-a ===
I'm sorry, I couldn't find any information about the minimum wage in Germany. However, I did find information about the minimum wage in Jordan.

According to Labor Law 1996, Article 52, a Tripartite Committee is responsible for determining the minimum wage in Jordan, taking into account the cost of living indicators issued by the competent official authorities. The decisions of the Committee are published in the Official Gazette.

Please note that this is general information and not legal advice. For specific legal advice, please consult a licensed lawyer.

=== openai-gpt-4o-mini ===
The documents provided do not contain any information about the minimum wage in Germany. They only address the minimum wage in Jordan as per the Labor Law 1996. For information specifically about Germany, you may need to consult other resources or legal texts.

This response is general information and not legal advice. For specific situations, please consult a licensed lawyer.

===

## LLM-as-judge: faithfulness and relevancy

The citation-matching checks above are proxies, not the real metrics -- worth being precise about the gap:
- **Faithfulness** (real definition): every claim in the answer is actually supported by the retrieved context, not just "the article number it cited happens to be in the context." A model can cite the right article and still misstate or embellish what that article says -- the regex check can't catch that, it only looks at numbers.
- **Relevancy** (real definition): the answer actually addresses the question asked -- on-topic, complete, not padded with tangents. Nothing above measured this at all.

Both require actually reading the answer against the source text, which regex can't do -- this needs an LLM judge. Judge model: **Claude Sonnet 5** (`anthropic/claude-sonnet-5`, via OpenRouter) -- deliberately *not* one of the three candidates being judged (Cohere/OpenAI/Qwen), to avoid self-preference bias where a model rates its own family's outputs more favorably.

For every (question, model) pair, the judge gets: the question, the exact retrieved context that model was given, and that model's answer -- then scores Faithfulness 1-5 and Relevancy 1-5 with a one-sentence reason each, returned as JSON.

In [6]:
import re as _re
import time as _time

from openai import OpenAI

judge_client = OpenAI(api_key=__import__("os").environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import (JUDGE_MODEL, JUDGE_PROMPT,
                     format_context_for_judge, make_judge)

judge = make_judge(judge_client)
judge = make_judge(judge_client)




judge_rows = []
for q in test_questions:
    chunks = chunks_by_qid[q["question_id"]]
    for model_name in generators:
        answer = answers[(q["question_id"], model_name)]
        score = judge(q["question_text"], chunks, answer)
        judge_rows.append({"question_id": q["question_id"], "model": model_name, **score})

judge_results = pd.DataFrame(judge_rows)
judge_results.to_csv(PROJECT / "eval" / "llm_comparison_judge_scores.csv", index=False, encoding="utf-8-sig")
print("done,", len(judge_results), "judge scores")

done, 33 judge scores


In [7]:
judge_summary = judge_results.groupby("model").agg(
    mean_faithfulness=("faithfulness_score", "mean"),
    mean_relevancy=("relevancy_score", "mean"),
    min_faithfulness=("faithfulness_score", "min"),
    min_relevancy=("relevancy_score", "min"),
).reset_index()
print(judge_summary.round(2).to_string(index=False))
print()
print("=== any score <= 3 (worth reading) ===")
low = judge_results[(judge_results["faithfulness_score"] <= 3) | (judge_results["relevancy_score"] <= 3)]
for _, row in low.iterrows():
    print(f"{row['question_id']} / {row['model']}: faithfulness={row['faithfulness_score']} ({row['faithfulness_reason']}) | relevancy={row['relevancy_score']} ({row['relevancy_reason']})")

             model  mean_faithfulness  mean_relevancy  min_faithfulness  min_relevancy
  cohere-command-a               4.82            4.73                 3              2
openai-gpt-4o-mini               5.00            4.91                 5              4
      qwen-2.5-72b               4.64            4.91                 3              4

=== any score <= 3 (worth reading) ===
q12 / qwen-2.5-72b: faithfulness=3 (The core legal claim is accurate per Article 62, but the response includes a stray Chinese sentence that is an unexplained artifact not derived from the sources.) | relevancy=4 (It directly answers both parts of the question (可否分批 and minimum duration) correctly, though the odd Chinese text detracts from clarity and completeness.)
q37 / cohere-command-a: faithfulness=3 (The definition quoted is accurate and correctly sourced to Article 2, but the answer erroneously attributes it to Article 3 first, creating a contradictory/incorrect citation.) | relevancy=5 (The answer 

## Results & decision

### Citation-matching proxies (regex-based)

| Model | Correct article rate (10 q) | Fabricated citations | Cost/question | Out-of-domain honesty |
|---|---|---|---|---|
| Cohere `command-a-03-2025` | 9/10 (90%) | 1 (q21) | ~$0.007 | Pass -- declined, no fabricated German figure |
| **OpenAI `gpt-4o-mini`** | **10/10 (100%)** | **0** | **~$0.0004** | Pass -- cleanest refusal, cited nothing beyond the given docs |
| Qwen `2.5-72b` (via OpenRouter/DeepInfra) | 9/10 (90%) | 1 (q21) | ~$0.0008 | Pass -- most terse refusal |

Two misses, both real:
- **Cohere missed q37** ("what counts as a profession") -- cited Article 3 instead of the correct Article 2, while both other models got it right.
- **Qwen missed q39** (is the old 1999 law still in effect) -- cited Article 11 instead of the correct Article 10.
- **Both Cohere and Qwen fabricated a citation on q21** (unemployment benefit calculation) -- both mentioned Article 90 (pension indexation, a different provision) alongside the correct Article 53, and Article 90 wasn't even in the 5 chunks retrieved for that query. GPT-4o-mini cited only the correct Article 53, nothing extra.

### LLM-as-judge: faithfulness and relevancy (the real metrics, not proxies)

| Model | Mean faithfulness (min) | Mean relevancy (min) |
|---|---|---|
| Cohere `command-a-03-2025` | 4.82 (3) | 4.73 (**2**) |
| **OpenAI `gpt-4o-mini`** | **5.00 (5)** | 4.91 (4) |
| Qwen `2.5-72b` | 4.64 (3) | 4.91 (4) |

Two findings the citation-matching check could never have caught:
- **Cohere's relevancy dropped to 2/5 on the out-of-domain trap question** -- the judge flagged that volunteering unrelated Jordan minimum-wage info instead of just declining "substitutes unrelated information" for what was actually asked. This confirms the "proactively helpful but off-target" behavior noted informally back in `07. generation_model_choice.ipynb` was a real relevancy defect, not just a stylistic quirk.
- **Qwen produced a stray Chinese sentence** in its answer to q12 (annual leave splitting) -- an outright generation-quality bug with nothing to do with citations, dragging its faithfulness score to 3/5. This is a genuine production risk specific to the cheap open-weight route that the citation check was structurally blind to.
- Cohere's q37 miscitation (Article 3 instead of 2) also surfaces here as a faithfulness ding (3/5) -- consistent with the regex finding, now with an explicit reason from an independent reader.

**GPT-4o-mini wins on every metric measured** -- perfect citation correctness, zero fabrication, the best (and most consistent -- never dropped below 5) faithfulness score, competitive relevancy, and by far the cheapest. It loses Cohere's native structured-citation API (manual citation-prompting instead), but empirically that trade cost nothing here.

Caveat: 10 questions is real signal, not proof -- worth confirming on the full Phase 8 eval once a model is picked.

**Decision: production generation is `gpt-4o-mini`, routed through OpenRouter rather than directly through OpenAI** (`openai/gpt-4o-mini`, same $0.15/$0.60 per-1M-token price as calling OpenAI directly -- OpenRouter passes through provider token rates with no markup, confirmed earlier). Routing through OpenRouter means all of this project's chat-model traffic (this and the Qwen comparison) goes through one vendor/key instead of juggling separate OpenAI and OpenRouter keys for no benefit. Sanity-checked working end-to-end outside this notebook before wiring in.

Implementation: `src/generation.py`'s `get_production_generator()` is the single entry point Phase 8 (eval) and Phase 9 (backend) should call. `Generator` (Cohere) stays in the file for this comparison but is no longer the production path.